# Music GPT Generation Notebook

Generate MIDI from your trained checkpoint (`best.pt` or `last.pt`).

Workflow:
1. Set config paths and sampling knobs.
2. Load checkpoint and model.
3. Sample token IDs autoregressively.
4. Decode tokens -> notes -> MIDI.
5. Save `.mid` to disk.

In [ ]:
from __future__ import annotations

import importlib.util
import json
from pathlib import Path

import pretty_midi
import torch
import torch.nn.functional as F

from train_gpt import MusicGPT, TrainConfig

# Load local src/tokenize.py explicitly to avoid stdlib tokenize conflict.
TOKENIZE_PATH = Path("tokenize.py").resolve()
spec = importlib.util.spec_from_file_location("music_tokenize", TOKENIZE_PATH)
if spec is None or spec.loader is None:
    raise ImportError(f"Could not load tokenize module from {TOKENIZE_PATH}")
music_tokenize = importlib.util.module_from_spec(spec)
spec.loader.exec_module(music_tokenize)
decode_tokens_to_notes = music_tokenize.decode_tokens_to_notes

In [ ]:
# --- Config ---
PROJECT_ROOT = Path("..").resolve()  # notebook is in src/
ID_TO_TOKEN_PATH = PROJECT_ROOT / "tokenized" / "id_to_token.json"
CHECKPOINT_PATH = PROJECT_ROOT / "src" / "best.pt"  # swap to last.pt if needed
OUTPUT_DIR = PROJECT_ROOT / "generated"
OUTPUT_NAME = "sample_from_best.midi"

# Default sampling knobs
MAX_NEW_TOKENS = 1200
TEMPERATURE = 1.0
TOP_K = 40
EOS_TOKEN = "EOS"
MAX_GENERATION_RETRIES = 3

# MIDI rendering knobs
TIME_STEP_SECONDS = 0.125
VELOCITY_BINS = 8
MIDI_PROGRAM = 0  # acoustic grand piano
MIN_NOTES_FOR_PLAYABLE = 8

# Mood presets (heuristic controls over randomness/length)
MOOD_PRESETS = {
    "neutral": {"temperature": 1.0, "top_k": 40, "max_new_tokens": 1200},
    "happy": {"temperature": 1.05, "top_k": 55, "max_new_tokens": 1400},
    "sad": {"temperature": 0.85, "top_k": 28, "max_new_tokens": 1300},
    "calm": {"temperature": 0.78, "top_k": 20, "max_new_tokens": 1500},
    "dark": {"temperature": 0.92, "top_k": 32, "max_new_tokens": 1400},
    "dramatic": {"temperature": 1.12, "top_k": 70, "max_new_tokens": 1700},
    "energetic": {"temperature": 1.15, "top_k": 75, "max_new_tokens": 1600},
}

mood_input = input(
    "Choose mood [neutral/happy/sad/calm/dark/dramatic/energetic] (Enter=neutral): "
).strip().lower()
SELECTED_MOOD = mood_input if mood_input in MOOD_PRESETS else "neutral"
preset = MOOD_PRESETS[SELECTED_MOOD]

TEMPERATURE = preset["temperature"]
TOP_K = preset["top_k"]
MAX_NEW_TOKENS = preset["max_new_tokens"]
OUTPUT_NAME = f"sample_{SELECTED_MOOD}_best.midi"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Checkpoint exists:", CHECKPOINT_PATH.exists())
print("id_to_token exists:", ID_TO_TOKEN_PATH.exists())
print("Selected mood:", SELECTED_MOOD)
print("Sampling -> temp:", TEMPERATURE, "top_k:", TOP_K, "max_new_tokens:", MAX_NEW_TOKENS)
print("Output MIDI:", OUTPUT_DIR / OUTPUT_NAME)

In [ ]:
# Load token mapping
id_to_token_raw = json.loads(ID_TO_TOKEN_PATH.read_text(encoding="utf-8"))
id_to_token = {int(k): v for k, v in id_to_token_raw.items()}
token_to_id = {v: k for k, v in id_to_token.items()}

print("Vocabulary size:", len(id_to_token))
print("BOS id:", token_to_id.get("BOS"), "EOS id:", token_to_id.get("EOS"))

In [ ]:
# Load model checkpoint
ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu")
cfg = TrainConfig(**ckpt["config"])

model = MusicGPT(cfg)
model.load_state_dict(ckpt["model_state"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

print("Loaded checkpoint:", CHECKPOINT_PATH.name)
print("Saved step:", ckpt.get("step", -1))
print("Device:", device)
print("Model max_seq_len:", cfg.max_seq_len)

In [ ]:
@torch.no_grad()
def generate_token_ids(
    model: MusicGPT,
    bos_id: int,
    eos_id: int,
    max_new_tokens: int,
    temperature: float,
    top_k: int | None,
    device: torch.device,
) -> list[int]:
    ids = [bos_id]

    for _ in range(max_new_tokens):
        idx = torch.tensor([ids], dtype=torch.long, device=device)
        idx_cond = idx[:, -model.cfg.max_seq_len :]

        logits = model(idx_cond)[:, -1, :]
        if temperature <= 0:
            next_id = int(torch.argmax(logits, dim=-1).item())
        else:
            logits = logits / temperature
            if top_k is not None and top_k > 0:
                values, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                kth = values[:, -1].unsqueeze(-1)
                logits = torch.where(logits < kth, torch.full_like(logits, float("-inf")), logits)
            probs = F.softmax(logits, dim=-1)
            next_id = int(torch.multinomial(probs, num_samples=1).item())

        ids.append(next_id)
        if next_id == eos_id:
            break

    return ids


def velocity_bin_to_midi_velocity(velocity_bin: int, num_bins: int = 8) -> int:
    # Center value of the bin mapped into MIDI velocity range [1, 127].
    value = int(round(((velocity_bin + 0.5) / num_bins) * 127))
    return max(1, min(127, value))


def notes_to_pretty_midi(
    notes,
    time_step_seconds: float,
    num_velocity_bins: int = 8,
    program: int = 0,
) -> pretty_midi.PrettyMIDI:
    pm = pretty_midi.PrettyMIDI()
    inst = pretty_midi.Instrument(program=program, is_drum=False, name="piano")

    for n in notes:
        inst.notes.append(
            pretty_midi.Note(
                velocity=velocity_bin_to_midi_velocity(n.velocity_bin, num_velocity_bins),
                pitch=int(n.pitch),
                start=float(n.start_step) * time_step_seconds,
                end=float(n.end_step) * time_step_seconds,
            )
        )

    pm.instruments.append(inst)
    return pm


def decode_tokens_to_notes_safe(tokens: list[str]):
    """
    Try strict decode first; if unmatched NOTE_ON remains, close dangling notes
    at the final observed time step so generation can still be rendered.
    """
    try:
        notes = decode_tokens_to_notes(tokens)
        return notes, {"used_fallback": False, "auto_closed_notes": 0}
    except ValueError:
        pass

    QuantizedNote = music_tokenize.QuantizedNote
    current_step = 0
    current_velocity_bin = 0
    active: dict[int, list[tuple[int, int]]] = {}
    notes = []

    for token in tokens:
        if token in {"PAD", "BOS", "EOS", "UNK"}:
            continue

        if token.startswith("TIME_SHIFT_"):
            shift = int(token.split("_")[-1])
            current_step += max(0, shift)
            continue

        if token.startswith("VELOCITY_"):
            current_velocity_bin = int(token.split("_")[-1])
            continue

        if token.startswith("NOTE_ON_"):
            pitch = int(token.split("_")[-1])
            active.setdefault(pitch, []).append((current_step, current_velocity_bin))
            continue

        if token.startswith("NOTE_OFF_"):
            pitch = int(token.split("_")[-1])
            starts = active.get(pitch, [])
            if not starts:
                continue
            start_step, velocity_bin = starts.pop(0)
            if current_step <= start_step:
                continue
            notes.append(
                QuantizedNote(
                    pitch=pitch,
                    velocity_bin=velocity_bin,
                    start_step=start_step,
                    end_step=current_step,
                )
            )

    auto_closed = 0
    final_step = current_step + 1
    for pitch, starts in active.items():
        for start_step, velocity_bin in starts:
            if final_step > start_step:
                notes.append(
                    QuantizedNote(
                        pitch=pitch,
                        velocity_bin=velocity_bin,
                        start_step=start_step,
                        end_step=final_step,
                    )
                )
                auto_closed += 1

    notes.sort(key=lambda n: (n.start_step, n.end_step, n.pitch, n.velocity_bin))
    return notes, {"used_fallback": True, "auto_closed_notes": auto_closed}

In [ ]:
# Generate token IDs (retry if output is too short to be musically useful)
bos_id = token_to_id["BOS"]
eos_id = token_to_id[EOS_TOKEN]

selected_ids = None
selected_tokens = None
selected_notes = None
selected_decode_info = None

for attempt in range(1, MAX_GENERATION_RETRIES + 1):
    gen_ids = generate_token_ids(
        model=model,
        bos_id=bos_id,
        eos_id=eos_id,
        max_new_tokens=MAX_NEW_TOKENS,
        temperature=TEMPERATURE,
        top_k=TOP_K,
        device=device,
    )
    gen_tokens = [id_to_token.get(i, "UNK") for i in gen_ids]
    notes, decode_info = decode_tokens_to_notes_safe(gen_tokens)

    print(
        f"Attempt {attempt}: tokens={len(gen_ids)}, notes={len(notes)}, "
        f"ended_with_eos={gen_ids[-1] == eos_id}, auto_closed={decode_info['auto_closed_notes']}"
    )

    if len(notes) >= MIN_NOTES_FOR_PLAYABLE:
        selected_ids = gen_ids
        selected_tokens = gen_tokens
        selected_notes = notes
        selected_decode_info = decode_info
        break

if selected_ids is None:
    # Keep the last attempt even if short.
    selected_ids = gen_ids
    selected_tokens = gen_tokens
    selected_notes = notes
    selected_decode_info = decode_info
    print("Warning: generated short sequence; saving last attempt anyway.")

print("Selected token count:", len(selected_ids))
print("Selected note count:", len(selected_notes))

In [ ]:
# Inspect selected generation
print("Decoded notes:", len(selected_notes))
print("Decode fallback used:", selected_decode_info["used_fallback"])
print("Auto-closed dangling notes:", selected_decode_info["auto_closed_notes"])
print("First 20 tokens:", selected_tokens[:20])

In [ ]:
# Render and save MIDI (.midi)
from datetime import datetime

pm = notes_to_pretty_midi(
    notes=selected_notes,
    time_step_seconds=TIME_STEP_SECONDS,
    num_velocity_bins=VELOCITY_BINS,
    program=MIDI_PROGRAM,
)

out_path = OUTPUT_DIR / OUTPUT_NAME
try:
    pm.write(str(out_path))
except PermissionError:
    stamped = f"{Path(OUTPUT_NAME).stem}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.midi"
    out_path = OUTPUT_DIR / stamped
    pm.write(str(out_path))
    print("Target file was locked; saved with fallback name.")

print("Saved:", out_path)
print("Total duration (s):", pm.get_end_time())
print("Playable note count:", len(pm.instruments[0].notes) if pm.instruments else 0)

## Tuning Tips

- More conservative (cleaner): `TEMPERATURE=0.8`, `TOP_K=20`
- More diverse: `TEMPERATURE=1.1`, `TOP_K=60`
- Longer pieces: increase `MAX_NEW_TOKENS`
- Try `CHECKPOINT_PATH = ... / "last.pt"` and compare outputs
- Output now defaults to `.midi` and retries generation if too few notes are produced

In [ ]:
# Batch-generate 10 playable MIDI samples
NUM_SAMPLES = 10
BATCH_PREFIX = "sample_best"

for i in range(NUM_SAMPLES):
    picked_ids = None
    picked_tokens = None
    picked_notes = None
    picked_info = None

    for _attempt in range(MAX_GENERATION_RETRIES):
        gen_ids = generate_token_ids(
            model=model,
            bos_id=token_to_id['BOS'],
            eos_id=token_to_id['EOS'],
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_k=TOP_K,
            device=device,
        )
        gen_tokens = [id_to_token.get(tid, 'UNK') for tid in gen_ids]
        notes, decode_info = decode_tokens_to_notes_safe(gen_tokens)
        picked_ids, picked_tokens, picked_notes, picked_info = gen_ids, gen_tokens, notes, decode_info

        if len(notes) >= MIN_NOTES_FOR_PLAYABLE:
            break

    pm = notes_to_pretty_midi(picked_notes, TIME_STEP_SECONDS, VELOCITY_BINS, MIDI_PROGRAM)
    out = OUTPUT_DIR / f"{BATCH_PREFIX}_{i+1:02d}.midi"
    pm.write(str(out))

    print(
        f"Saved: {out.name} | tokens={len(picked_ids)} | notes={len(picked_notes)} | "
        f"dur={pm.get_end_time():.2f}s | fallback={picked_info['used_fallback']} | "
        f"auto_closed={picked_info['auto_closed_notes']}"
    )